In [10]:
from pathlib import Path
from src.application.contracts import PipelineRequest
from src.application.pipeline import ViRAGEPipeline
from src.application.settings import ViRAGESettings
from rich import print as rprint

In [11]:
root = Path().resolve().parents[0]

In [12]:
settings = ViRAGESettings(artifact_root=root / "artifacts_test", allow_code_execution=True)

In [13]:
rprint(settings)

ViRAGESettings(
    artifact_root=WindowsPath('C:/Users/nikita/projects/mas_rag/artifacts_test'),
    project_name='ViRAGE',
    allow_code_execution=True,
    default_figure_dpi=144
)

In [14]:
pipeline = ViRAGEPipeline(settings=settings)

In [17]:
pipeline

In [18]:
print(pipeline.graph.get_graph().draw_ascii())

                      +-----------+                        
                      | __start__ |                        
                      +-----------+                        
                            *                              
                            *                              
                            *                              
                +---------------------+                    
                | query_understanding |                    
                +---------------------+                    
                   ..               ..                     
                ...                   ...                  
              ..                         ..                
+--------------------+          +------------------------+ 
| planning_canonical |          | planning_non_canonical | 
+--------------------+          +------------------------+ 
                   **               **                     
                     ***         ***    

In [19]:
settings = ViRAGESettings(artifact_root=root / "artifacts_test", allow_code_execution=True)
pipeline = ViRAGEPipeline(settings=settings)
result = pipeline.invoke(PipelineRequest(query="Analyze trend of sales over time",
                                         data_path=(root / "examples" / "demo.csv").as_posix()))

In [20]:
rprint(result)

PipelineResult(
    run_id='8806e74c3e28407c9bcc1fed65f6466c',
    query='Analyze trend of sales over time',
    data_path='C:/Users/nikita/projects/mas_rag/examples/demo.csv',
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    query_understanding=QueryUnderstandingResult(
        intent='Analyze trend of sales over time',
        requested_operations=['trend analysis'],
        candidate_charts=['line'],
        constraints=[],
        case_type=<ChartCaseType.CANONICAL: 'canonical'>,
        confidence=0.65
    ),
    planning=PlanningResult(
        mode=<ChartCaseType.CANONICAL: 'canonical'>,
        steps=[
            PlanningStep(name='profile_data', description='Inspect dataset structure and quality.'),
            PlanningStep(name='prepare_data', description='Apply minimal cleaning and normalization.'),
            PlanningStep(
                name='retrieve_visual_knowledge',
                description='Retrieve charting guidance for the task.'
            ),
            PlanningStep(name='generate_and_run', description='Generate and execute plotting code.'),
            PlanningStep(name='analyze_chart', description='Read chart structure and derive facts.'),
            PlanningStep(name='verify', description='Validate conclusions against artifacts and metrics.')
        ],
        success_criteria=[
            'At least one chart is generated successfully.',
            'Artifacts are saved and traceable.',
            'Final statements include evidence references.'
        ]
    ),
    data_profile=DataProfile(
        row_count=5,
        col_count=4,
        columns=[
            DataColumnProfile(name='date', dtype='str', missing_ratio=0.0, unique_count=5),
            DataColumnProfile(name='sales', dtype='int64', missing_ratio=0.0, unique_count=5),
            DataColumnProfile(name='region', dtype='str', missing_ratio=0.0, unique_count=2),
            DataColumnProfile(name='units', dtype='int64', missing_ratio=0.0, unique_count=5)
        ],
        likely_numeric_columns=['sales', 'units'],
        likely_categorical_columns=['date', 'region'],
        likely_time_columns=['date'],
        quality_notes=[]
    ),
    data_preparation=DataPreparationResult(
        output_path='C:/Users/nikita/projects/mas_rag/artifacts_test/8806e74c3e28407c9bcc1fed65f6466c/cleaned_data.
csv',
        operations=['to_datetime:date'],
        row_count=5,
        col_count=4
    ),
    visrag=VisRAGResult(
        recommendations=[
            VisRAGRecommendation(
                chart_family='line',
                rationale='Time-like field detected; line chart suits trend analysis.',
                priority=1
            )
        ],
        rules=['Use clear titles and axis labels.', 'Avoid overcrowded visuals.', 'Prefer readable defaults.'],
        caveats=[]
    ),
    codegen=CodegenResult(
        language='python',
        chart_type='line',
        code='\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport 
matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\n\ndef 
_safe_numeric(series: pd.Series):\n    return pd.to_numeric(series, errors="coerce")\n\n\ndef main(output_dir: str 
= "C:/Users/nikita/projects/mas_rag/artifacts_test/8806e74c3e28407c9bcc1fed65f6466c/execution") -> None:\n    out =
Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    df = 
pd.read_csv(r"C:/Users/nikita/projects/mas_rag/artifacts_test/8806e74c3e28407c9bcc1fed65f6466c/cleaned_data.csv")\n
chart_type = \'line\'\n    x_col = \'date\'\n    y_col = \'sales\'\n\n    fig, ax = plt.subplots(figsize=(8, 5), 
dpi=144)\n    title = \'Analyze trend of sales over time\'\n    metrics = {"row_count": int(len(df)), 
"column_count": int(len(df.columns)), "chart_type": chart_type}\n\n    if chart_type == "line" and y_col:\n        
x = pd.to_datetime(df[x_col]) if x_col else pd.RangeIndex(len(df))\n        y = _safe_numeric(df[y_col])\n        
ax.plot(x, y